# レッスン4: オーディオエフェクトとダイナミクス

前回まで学んだこと：
- 基本的な音響合成（発振器、エンベロープ）
- フィルターによる音色デザイン
- 楽器音の作成

今回のレッスンで学ぶこと：
- **リバーブ（残響）**で音に空間的な広がりを与える
- **ディレイ（遅延）**でエコー効果を作る
- **コーラス**で音を豊かにする
- **ダイナミクス**（音量制御）の基本

🎵 ゴール：プロっぽい音響効果を理解し、音楽に深みを与える技術を身につける

## 準備：ライブラリのセットアップ

⚡ **高速セットアップ**：初回実行時のみインストール、以降は軽量な更新のみ
- 既存環境がある場合：約10-15秒で完了
- 初回セットアップ：約1-2分で完了

In [ ]:
# 🛠️ 環境セットアップ
# どちらの環境でも同じノートブックが動作します

# 環境自動判定
try:
    import google.colab
    IN_COLAB = True
    print("🔧 Google Colabで実行中...")

    # Colabの場合：GitHubから最新版をクローン
    !git clone https://github.com/ggszk/simple-audio-programming.git
    import sys
    sys.path.append('/content/simple-audio-programming')
    print("✅ GitHubから最新版をセットアップ完了！")

except ImportError:
    IN_COLAB = False
    print("🏠 ローカル環境で実行中")
    print("✅ pyproject.tomlで管理された環境を使用")

# 🎵 オーディオライブラリの統一インポート
from audio_lib import (
    sine_wave, sawtooth_wave, square_wave, triangle_wave,
    adsr, linear_envelope, AudioSignal, save_audio,
    LowPassFilter, HighPassFilter,
    Reverb, Distortion, Delay, Chorus, Compressor,
    note_to_frequency, note_name_to_number,
)

from audio_lib.effects.filters import BandPassFilter

print("✅ すべてのエフェクトが利用可能です")

from IPython.display import Audio, display
import numpy as np
import matplotlib.pyplot as plt

# 日本語表示設定（Colab用）
if IN_COLAB:
    import japanize_matplotlib

import warnings
warnings.filterwarnings('ignore')

print("🎼 サンプリング周波数: 44100Hz")
print("🚀 セットアップ完了！エフェクトの世界へようこそ！")

## 4.1 リバーブ（残響）：音に空間を与える

リバーブは部屋や空間での音の反響をシミュレートする効果です。

- **コンサートホール**：長い残響
- **小さな部屋**：短い残響
- **屋外**：ほぼ残響なし

音楽では空間的な広がりや深みを演出するために使用されます。

In [ ]:
# シンプルなピアノ音を作成（前回の復習）
def create_simple_piano(note_freq, duration=2.0):
    """シンプルなピアノ音"""
    signal = sine_wave(note_freq, duration)

    # ピアノ風エンベロープ
    envelope = adsr(
        duration,
        attack=0.01,
        decay=0.3,
        sustain=0.4,
        release=1.5
    )

    # エンベロープを適用
    return signal * envelope

# ドライ音（リバーブなし）
dry_piano = create_simple_piano(note_to_frequency(note_name_to_number('C4')))

print("ドライ音（リバーブなし）:")
display(Audio(dry_piano.data, rate=dry_piano.sample_rate))

# リバーブありの音
reverb = Reverb(room_size=0.8, damping=0.5, wet_level=0.3)
wet_piano = reverb.process(dry_piano)

print("リバーブあり（大きな部屋）:")
display(Audio(wet_piano.data, rate=wet_piano.sample_rate))

### 🎧 聞き比べてみよう
- ドライ音：直接的で近い感じ
- リバーブあり：空間的で遠い感じ、音の尻尾が長い

### リバーブパラメータの効果

リバーブの音質は3つの主要パラメータで決まります：

#### 🏠 **room_size**（部屋のサイズ）
- **0.1-0.3**: 小さな部屋、練習室
- **0.4-0.6**: 中規模の部屋、スタジオ
- **0.7-0.8**: 大きなホール、教会

#### 🎛️ **damping**（減衰・吸音）
- **0.7-0.9**: 吸音材が多い（カーペット、カーテン）
- **0.4-0.6**: 適度な吸音（一般的な部屋）
- **0.1-0.3**: 反響しやすい（石の壁、空の部屋）

#### 🔊 **wet_level**（エフェクト音の量）
- **0.1-0.2**: 控えめな残響
- **0.3-0.4**: 適度な残響  
- **0.5-0.6**: しっかりとした残響

⚠️ **注意**: 極端な設定（room_size > 0.9, damping < 0.2）はノイズの原因になります

In [ ]:
# 異なるリバーブ設定を比較
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('A4')), 1.5)

# 設定1：小さな部屋
reverb_small = Reverb(room_size=0.3, damping=0.8, wet_level=0.2)
small_room = reverb_small.process(base_sound)

print("小さな部屋（room_size=0.3, damping=0.8）:")
display(Audio(small_room.data, rate=small_room.sample_rate))

# 設定2：中規模ホール（安定した設定）
reverb_hall = Reverb(room_size=0.7, damping=0.4, wet_level=0.4)
concert_hall = reverb_hall.process(base_sound)

print("コンサートホール（room_size=0.7, damping=0.4）:")
display(Audio(concert_hall.data, rate=concert_hall.sample_rate))

# 設定3：大きなホール（より安定した設定）
reverb_large = Reverb(room_size=0.8, damping=0.3, wet_level=0.5)
large_hall = reverb_large.process(base_sound)

print("大きなホール（room_size=0.8, damping=0.3）:")
display(Audio(large_hall.data, rate=large_hall.sample_rate))

print("\n💡 リバーブパラメータの効果:")
print("- room_size: 部屋の大きさ（0.3=小部屋、0.8=大ホール）")
print("- damping: 減衰率（0.8=よく減衰、0.3=長く響く）")
print("- wet_level: エフェクト音の量（0.2=控えめ、0.5=しっかり）")

## 4.2 ディレイ（遅延）：エコー効果を作る

ディレイは音を一定時間遅らせて元の音に重ねる効果です。

- **短いディレイ**：音の厚みや広がり
- **長いディレイ**：明確なエコー効果
- **フィードバック**：エコーの繰り返し

In [ ]:
# ディレイ効果を試してみよう
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('E4')), 1.0)

print("元の音:")
display(Audio(base_sound.data, rate=base_sound.sample_rate))

# 短いディレイ（音の厚み）
delay_short = Delay(delay_time=0.1, feedback=0.3, wet_level=0.4)
short_delay_sound = delay_short.process(base_sound)

print("短いディレイ（0.1秒）:")
display(Audio(short_delay_sound.data, rate=short_delay_sound.sample_rate))

# 長いディレイ（明確なエコー）
delay_long = Delay(delay_time=0.5, feedback=0.4, wet_level=0.5)
long_delay_sound = delay_long.process(base_sound)

print("長いディレイ（0.5秒）:")
display(Audio(long_delay_sound.data, rate=long_delay_sound.sample_rate))

# フィードバック多め（エコーの繰り返し）
delay_feedback = Delay(delay_time=0.3, feedback=0.6, wet_level=0.4)
feedback_sound = delay_feedback.process(base_sound)

print("フィードバック多め（エコーの繰り返し）:")
display(Audio(feedback_sound.data, rate=feedback_sound.sample_rate))

## 4.3 コーラス：音を豊かにする

コーラスは音を少しずつピッチや時間をずらしてコピーし、重ね合わせる効果です。
まるで複数の楽器や声で演奏しているような豊かさを生み出します。

### コーラスパラメータの重要なポイント：

#### 🎵 **depth（変調の深さ）**
- **0.01-0.03**: 自然で控えめな効果
- **0.04-0.06**: しっかりとした効果
- **0.07以上**: 過度になり「ぴよぴよ」した不自然な音になりがち

#### 📊 **rate（変調の速度）**
- **0.5-2.0 Hz**: ゆっくりとした効果
- **2.0-4.0 Hz**: 一般的な速度
- **5.0以上**: 速いビブラート的効果

#### 🔊 **wet_level（エフェクト音の量）**
- **0.2-0.4**: 控えめ
- **0.4-0.6**: 適度
- **0.6以上**: 強い効果

⚠️ **注意**: depthが大きすぎると音程が不安定になり、聞き苦しくなります

In [ ]:
# コーラス効果を試してみよう
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('G4')), 2.0)

print("元の音:")
display(Audio(base_sound.data, rate=base_sound.sample_rate))

# 軽いコーラス（自然な効果）
chorus_light = Chorus(rate=2.0, depth=0.015, wet_level=0.4)
light_chorus_sound = chorus_light.process(base_sound)

print("軽いコーラス:")
display(Audio(light_chorus_sound.data, rate=light_chorus_sound.sample_rate))

# 深いコーラス（しっかりとした効果）
chorus_deep = Chorus(rate=1.5, depth=0.025, wet_level=0.6)
deep_chorus_sound = chorus_deep.process(base_sound)

print("深いコーラス:")
display(Audio(deep_chorus_sound.data, rate=deep_chorus_sound.sample_rate))

# 速いコーラス（ビブラート的）
chorus_fast = Chorus(rate=5.0, depth=0.01, wet_level=0.5)
fast_chorus_sound = chorus_fast.process(base_sound)

print("速いコーラス（ビブラート的）:")
display(Audio(fast_chorus_sound.data, rate=fast_chorus_sound.sample_rate))

## 4.4 エフェクトの組み合わせ：プロフェッショナルな音作り

In [ ]:
def create_lush_piano(note_freq, duration=3.0):
    """豊かなエフェクトをかけたピアノ音"""

    # 基本のピアノ音
    base_sound = create_simple_piano(note_freq, duration)

    # エフェクトチェーン：コーラス → ディレイ → リバーブ
    chorus = Chorus(rate=2.5, depth=0.02, wet_level=0.3)
    delay = Delay(delay_time=0.15, feedback=0.25, wet_level=0.2)
    reverb = Reverb(room_size=0.6, wet_level=0.3)

    # エフェクトを順番に適用
    sound = chorus.process(base_sound)
    sound = delay.process(sound)
    sound = reverb.process(sound)

    return sound

# 比較してみよう
dry_sound = create_simple_piano(note_to_frequency(note_name_to_number('C4')), 3.0)
lush_sound = create_lush_piano(note_to_frequency(note_name_to_number('C4')), 3.0)

print("ドライなピアノ音:")
display(Audio(dry_sound.data, rate=dry_sound.sample_rate))

print("豊かなエフェクト付きピアノ音:")
display(Audio(lush_sound.data, rate=lush_sound.sample_rate))

## 4.5 ダイナミクス処理の基本

ダイナミクスは音量の変化を制御する技術です。

- **コンプレッサー**：音量差を小さくする
- **リミッター**：音量の上限を設定
- **ゲート**：小さな音をカット

In [ ]:
# シンプルなコンプレッサー効果を実装
def simple_compressor(signal, threshold=0.7, ratio=4.0):
    """シンプルなコンプレッサー

    Args:
        signal: 入力音声（AudioSignal）
        threshold: コンプレッションが始まるレベル
        ratio: 圧縮比（4.0なら4:1の圧縮）
    """
    compressed = signal.data.copy()

    # 各サンプルに対してコンプレッション適用
    for i in range(len(compressed)):
        sample_level = abs(compressed[i])

        if sample_level > threshold:
            # 閾値を超えた分を圧縮
            excess = sample_level - threshold
            compressed_excess = excess / ratio
            new_level = threshold + compressed_excess

            # 元の符号を保持
            if compressed[i] >= 0:
                compressed[i] = new_level
            else:
                compressed[i] = -new_level

    return AudioSignal(compressed, signal.sample_rate)

# 動的な音量変化を持つ音を作成
def create_dynamic_sound(duration=4.0):
    """音量が変化する音を作成"""
    freq = note_to_frequency(note_name_to_number('A4'))
    signal = sine_wave(freq, duration)

    # 時間とともに変化する音量エンベロープ
    time_samples = len(signal)
    volume_envelope = np.ones(time_samples)

    # 不規則な音量変化を作成
    for i in range(time_samples):
        t = i / time_samples
        # 複数の正弦波で複雑な音量変化
        volume = 0.5 + 0.3 * np.sin(t * 8 * np.pi) + 0.2 * np.sin(t * 20 * np.pi)
        volume_envelope[i] = max(0.1, min(1.0, volume))

    return AudioSignal(signal.data * volume_envelope, signal.sample_rate)

# ダイナミクス処理の比較
dynamic_sound = create_dynamic_sound()
compressed_sound = simple_compressor(dynamic_sound, threshold=0.6, ratio=3.0)

print("元の音（音量変化あり）:")
display(Audio(dynamic_sound.data, rate=dynamic_sound.sample_rate))

print("コンプレッサー適用後（音量変化を抑制）:")
display(Audio(compressed_sound.data, rate=compressed_sound.sample_rate))

# 波形の比較
plt.figure(figsize=(12, 6))
time = np.linspace(0, 4, len(dynamic_sound))

plt.subplot(2, 1, 1)
plt.plot(time, dynamic_sound.data, alpha=0.7)
plt.title('元の音（音量変化あり）')
plt.ylabel('振幅')
plt.grid(True, alpha=0.3)

plt.subplot(2, 1, 2)
plt.plot(time, compressed_sound.data, alpha=0.7, color='orange')
plt.title('コンプレッサー適用後')
plt.xlabel('時間 (秒)')
plt.ylabel('振幅')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# パフォーマンステクニック：エフェクトの時間変化
base_note = note_to_frequency(note_name_to_number('D4'))
base_sound = create_simple_piano(base_note, 5.0)

# エフェクトを時間で変化させる
print("エフェクトの時間変化:")

# 1. だんだん強くなるコーラス
chorus = Chorus(rate=3.0, depth=0.02, wet_level=0.5)
chorus_sound = chorus.process(base_sound)
display(Audio(chorus_sound.data, rate=chorus_sound.sample_rate))

# 2. 途中で変化するディレイ（手動で作成）
delay1 = Delay(delay_time=0.1, feedback=0.3, wet_level=0.2)
delay2 = Delay(delay_time=0.25, feedback=0.4, wet_level=0.4)

# 前半はdelay1、後半はdelay2
split_point = len(base_sound) // 2
part1 = delay1.process(AudioSignal(base_sound.data[:split_point], base_sound.sample_rate))
part2 = delay2.process(AudioSignal(base_sound.data[split_point:], base_sound.sample_rate))

# パート1をそのままの長さで、パート2を接続
combined_data = np.concatenate([part1.data, part2.data[:len(base_sound)-len(part1)]])
combined_delay = AudioSignal(combined_data, base_sound.sample_rate)

print("変化するディレイ:")
display(Audio(combined_delay.data, rate=combined_delay.sample_rate))

# パフォーマンステクニック：異なるコーラス設定
base_note = note_to_frequency(note_name_to_number('D4'))
base_sound = create_simple_piano(base_note, 3.0)

print("=== コーラス効果の比較 ===")

print("1. 元の音（エフェクトなし）:")
display(Audio(base_sound.data, rate=base_sound.sample_rate))

print("2. 軽いコーラス (depth=0.01):")
chorus_light = Chorus(rate=2.0, depth=0.01, wet_level=0.3)
light_sound = chorus_light.process(base_sound)
display(Audio(light_sound.data, rate=light_sound.sample_rate))

print("3. 標準コーラス (depth=0.02):")
chorus_standard = Chorus(rate=2.5, depth=0.02, wet_level=0.4)
standard_sound = chorus_standard.process(base_sound)
display(Audio(standard_sound.data, rate=standard_sound.sample_rate))

print("4. しっかりコーラス (depth=0.03):")
chorus_rich = Chorus(rate=1.8, depth=0.03, wet_level=0.5)
rich_sound = chorus_rich.process(base_sound)
display(Audio(rich_sound.data, rate=rich_sound.sample_rate))

print("💡 音の変化を聞き比べてみましょう:")
print("- depth が小さい → 自然で控えめな効果")
print("- depth が大きい → より明確だが不自然になりがち")
print("- 0.01-0.03 程度が実用的です")

## 4.6 実践演習：音楽フレーズにエフェクトを適用

In [ ]:
# 音楽ジャンル別エフェクト設定
base_melody = [
    note_to_frequency(note_name_to_number('C4')),
    note_to_frequency(note_name_to_number('E4')),
    note_to_frequency(note_name_to_number('G4')),
    note_to_frequency(note_name_to_number('C5'))
]

def create_melody(notes, note_duration=1.0):
    """メロディを作成"""
    sounds = []
    for note_freq in notes:
        note_sound = create_simple_piano(note_freq, note_duration)
        sounds.append(note_sound.data)
    return AudioSignal(np.concatenate(sounds), 44100)

base_sound = create_melody(base_melody, 1.5)

print("1. クラシック（リバーブ重視）")
# クラシック：自然なリバーブ
reverb_classic = Reverb(room_size=0.8, wet_level=0.5)
classic_sound = reverb_classic.process(base_sound)
display(Audio(classic_sound.data, rate=classic_sound.sample_rate))

print("2. ポップス（適度なコーラス）")
# ポップス：コーラスで豊かさを
chorus = Chorus(rate=2.0, depth=0.015, wet_level=0.3)
delay = Delay(delay_time=0.125, feedback=0.2, wet_level=0.2)
pop_sound = chorus.process(base_sound)
pop_sound = delay.process(pop_sound)
display(Audio(pop_sound.data, rate=pop_sound.sample_rate))

print("3. アンビエント（深いリバーブとディレイ）")
# アンビエント：空間的な効果
reverb_ambient = Reverb(room_size=0.9, wet_level=0.7)
delay_ambient = Delay(delay_time=0.5, feedback=0.6, wet_level=0.4)
ambient_sound = reverb_ambient.process(base_sound)
ambient_sound = delay_ambient.process(ambient_sound)
display(Audio(ambient_sound.data, rate=ambient_sound.sample_rate))

## 4.7 チャレンジ課題

### 🎯 課題1：自分だけのエフェクトプリセット
異なる音楽ジャンルに適したエフェクト設定を作ってみよう

In [ ]:
def create_genre_preset(base_sound, genre="pop"):
    """ジャンル別エフェクトプリセット"""

    if genre == "pop":
        # ポップス：適度なコーラスと短いリバーブ
        chorus = Chorus(rate=2.0, depth=0.015, wet_level=0.3)
        reverb = Reverb(room_size=0.5, damping=0.6, wet_level=0.2)

        sound = chorus.process(base_sound)
        sound = reverb.process(sound)

    elif genre == "ambient":
        # アンビエント：深いリバーブと長いディレイ
        delay = Delay(delay_time=0.6, feedback=0.5, wet_level=0.4)
        reverb = Reverb(room_size=0.9, damping=0.2, wet_level=0.6)
        chorus = Chorus(rate=1.0, depth=0.06, wet_level=0.4)

        sound = chorus.process(base_sound)
        sound = delay.process(sound)
        sound = reverb.process(sound)

    elif genre == "dry":
        # ドライ：軽いコンプレッションのみ
        sound = simple_compressor(base_sound, threshold=0.8, ratio=2.0)

    else:  # デフォルト
        sound = base_sound

    return sound

# テスト用の音
test_sound = create_simple_piano(note_to_frequency(note_name_to_number('A4')), 2.0)

# 各ジャンルプリセットを試す
genres = ["dry", "pop", "ambient"]

for genre in genres:
    processed_sound = create_genre_preset(test_sound, genre)
    print(f"{genre.upper()}プリセット:")
    display(Audio(processed_sound.data, rate=processed_sound.sample_rate))
    print()

def create_dynamic_effects_demo():
    """動的にエフェクトが変化するデモ"""

    # ベースとなる長い音
    base_freq = note_to_frequency(note_name_to_number('F3'))
    long_sound = create_simple_piano(base_freq, 8.0)

    # 時間とともに変化するパラメータ
    # 0-2秒：ドライ
    # 2-4秒：コーラス追加
    # 4-6秒：ディレイ追加
    # 6-8秒：リバーブ追加

    sample_rate = long_sound.sample_rate
    total_samples = len(long_sound)

    # 各セクションの長さ
    section_length = total_samples // 4

    # セクション1: ドライ（原音）
    section1 = long_sound.data[:section_length]

    # セクション2: コーラス
    chorus = Chorus(rate=2.0, depth=0.015, wet_level=0.3)
    section2 = chorus.process(AudioSignal(long_sound.data[section_length:section_length*2], sample_rate))

    # セクション3: コーラス + ディレイ
    delay = Delay(delay_time=0.15, feedback=0.3, wet_level=0.25)
    section3_base = chorus.process(AudioSignal(long_sound.data[section_length*2:section_length*3], sample_rate))
    section3 = delay.process(section3_base)

    # セクション4: フルエフェクト（コーラス + ディレイ + リバーブ）
    reverb = Reverb(room_size=0.6, wet_level=0.4)
    section4_base = chorus.process(AudioSignal(long_sound.data[section_length*3:], sample_rate))
    section4_base = delay.process(section4_base)
    section4 = reverb.process(section4_base)

    # 全セクションを結合
    dynamic_data = np.concatenate([section1, section2.data, section3.data, section4.data])

    return AudioSignal(dynamic_data, sample_rate)

# 動的エフェクトデモを実行
print("動的エフェクト変化のデモ:")

dynamic_demo = create_dynamic_effects_demo()
display(Audio(dynamic_demo.data, rate=dynamic_demo.sample_rate))

### 🎯 課題2：エフェクトの順序による違い
エフェクトを適用する順序で音がどう変わるか実験してみよう

In [ ]:
# エフェクトチェーンの順序比較
base_sound = create_simple_piano(note_to_frequency(note_name_to_number('C4')), 2.0)

# チェーン1：リバーブ → ディレイ
reverb = Reverb(room_size=0.7, damping=0.4, wet_level=0.4)
delay = Delay(delay_time=0.3, feedback=0.3, wet_level=0.4)

chain1 = reverb.process(base_sound)
chain1 = delay.process(chain1)

print("チェーン1（リバーブ → ディレイ）:")
display(Audio(chain1.data, rate=chain1.sample_rate))

# チェーン2：ディレイ → リバーブ
chain2 = delay.process(base_sound)
chain2 = reverb.process(chain2)

print("チェーン2（ディレイ → リバーブ）:")
display(Audio(chain2.data, rate=chain2.sample_rate))

print("🎧 違いを聞き比べてみよう！")
print("- チェーン1：エコーに残響がかかる")
print("- チェーン2：残響にエコーがかかる")

# エフェクトパラメータの実験
base_note = create_simple_piano(note_to_frequency(note_name_to_number('A4')), 3.0)

print("=== エフェクトパラメータ比較 ===")

# 1. コーラス深度の比較
print("1. コーラス深度の違い:")

print("   浅いコーラス (depth=0.01):")
chorus_shallow = Chorus(rate=2.0, depth=0.01, wet_level=0.4)
shallow_sound = chorus_shallow.process(base_note)
display(Audio(shallow_sound.data, rate=shallow_sound.sample_rate))

print("   標準コーラス (depth=0.02):")
chorus_standard = Chorus(rate=2.0, depth=0.02, wet_level=0.4)
standard_sound = chorus_standard.process(base_note)
display(Audio(standard_sound.data, rate=standard_sound.sample_rate))

print("   深いコーラス (depth=0.03):")
chorus_deep = Chorus(rate=2.0, depth=0.03, wet_level=0.4)
deep_sound = chorus_deep.process(base_note)
display(Audio(deep_sound.data, rate=deep_sound.sample_rate))

# 2. ディレイフィードバックの比較
print("2. ディレイフィードバックの違い:")

print("   軽いフィードバック (0.2):")
delay_light = Delay(delay_time=0.2, feedback=0.2, wet_level=0.3)
light_delay = delay_light.process(base_note)
display(Audio(light_delay.data, rate=light_delay.sample_rate))

print("   強いフィードバック (0.5):")
delay_strong = Delay(delay_time=0.2, feedback=0.5, wet_level=0.3)
strong_delay = delay_strong.process(base_note)
display(Audio(strong_delay.data, rate=strong_delay.sample_rate))

### 🎯 課題3：ライブパフォーマンス風のエフェクト切り替え

In [ ]:
# 時間軸でエフェクトが変化する音を作成
def create_performance_sound():
    """ライブパフォーマンス風：時間とともにエフェクトが変化"""

    # 4つのセクション、それぞれ2秒
    sections = []
    base_freq = note_to_frequency(note_name_to_number('G4'))

    # セクション1：ドライ
    section1 = create_simple_piano(base_freq, 2.0)
    sections.append(section1.data)

    # セクション2：コーラス追加（適切なdepth値）
    section2 = create_simple_piano(base_freq, 2.0)
    chorus = Chorus(rate=3.0, depth=0.015, wet_level=0.5)  # depth値を調整
    section2 = chorus.process(section2)
    sections.append(section2.data)

    # セクション3：ディレイ追加
    section3 = create_simple_piano(base_freq, 2.0)
    section3 = chorus.process(section3)
    delay = Delay(delay_time=0.25, feedback=0.4, wet_level=0.4)
    section3 = delay.process(section3)
    sections.append(section3.data)

    # セクション4：フルエフェクト
    section4 = create_simple_piano(base_freq, 2.0)
    section4 = chorus.process(section4)
    section4 = delay.process(section4)
    reverb = Reverb(room_size=0.8, damping=0.3, wet_level=0.5)
    section4 = reverb.process(section4)
    sections.append(section4.data)

    # 全セクションを連結
    return AudioSignal(np.concatenate(sections), 44100)

performance_sound = create_performance_sound()

print("ライブパフォーマンス風エフェクト変化：")
print("0-2秒：ドライ → 2-4秒：コーラス → 4-6秒：ディレイ追加 → 6-8秒：フルエフェクト")
display(Audio(performance_sound.data, rate=performance_sound.sample_rate))

# 【総合演習】あなただけのエフェクトプリセットを作ろう！

def create_my_preset(base_sound, preset_name="My Preset"):
    """
    あなただけのエフェクトプリセットを作成する関数

    パラメータを変更して、好みの音作りをしてみましょう！
    """

    # ここでパラメータを調整してください
    # 以下は例です。自由に変更してください！

    if preset_name == "Dream Pad":
        # 夢のような音色
        chorus = Chorus(rate=1.5, depth=0.015, wet_level=0.4)
        delay = Delay(delay_time=0.3, feedback=0.4, wet_level=0.3)
        reverb = Reverb(room_size=0.8, wet_level=0.6)

    elif preset_name == "Space Echo":
        # 宇宙的なエコー
        chorus = Chorus(rate=3.0, depth=0.02, wet_level=0.3)
        delay = Delay(delay_time=0.5, feedback=0.6, wet_level=0.5)
        reverb = Reverb(room_size=0.9, wet_level=0.4)

    elif preset_name == "Warm Piano":
        # 温かいピアノ
        chorus = Chorus(rate=2.0, depth=0.01, wet_level=0.2)
        delay = Delay(delay_time=0.1, feedback=0.2, wet_level=0.15)
        reverb = Reverb(room_size=0.5, wet_level=0.3)

    else:
        # デフォルト（あなたのオリジナル設定を作ってください！）
        chorus = Chorus(rate=2.5, depth=0.02, wet_level=0.35)
        delay = Delay(delay_time=0.2, feedback=0.3, wet_level=0.25)
        reverb = Reverb(room_size=0.6, wet_level=0.4)

    # エフェクトを順番に適用
    sound = chorus.process(base_sound)
    sound = delay.process(sound)
    sound = reverb.process(sound)

    return sound

# テスト用の音を作成
test_note = create_simple_piano(note_to_frequency(note_name_to_number('C4')), 4.0)

# プリセットを試してみよう
print("🎹 様々なプリセットを試してみましょう！")

print("1. Dream Pad:")
dream_sound = create_my_preset(test_note, "Dream Pad")
display(Audio(dream_sound.data, rate=dream_sound.sample_rate))

print("2. Space Echo:")
space_sound = create_my_preset(test_note, "Space Echo")
display(Audio(space_sound.data, rate=space_sound.sample_rate))

print("3. Warm Piano:")
warm_sound = create_my_preset(test_note, "Warm Piano")
display(Audio(warm_sound.data, rate=warm_sound.sample_rate))

print("4. あなたのオリジナル:")
my_sound = create_my_preset(test_note, "My Original")
display(Audio(my_sound.data, rate=my_sound.sample_rate))

print("""
💡 チャレンジ：
1. 上記の関数のelse部分を編集して、あなただけのプリセットを作ってください
2. パラメータを変更して音の違いを確認してください
3. 複数のプリセットを組み合わせることもできます！

🎵 パラメータのヒント：
- chorus.depth: 0.005-0.03 (小さいほど自然)
- delay.feedback: 0.1-0.7 (大きいほどエコーが長い)
- reverb.room_size: 0.3-0.9 (大きいほど広い空間)
- wet_level: 0.1-0.8 (エフェクトの強さ)
""")

## まとめ

### 今回学んだこと：
1. **リバーブ**：空間的な広がりと深みを与える
2. **ディレイ**：エコー効果と音の厚みを作る
3. **コーラス**：音を豊かにし、複数楽器感を演出
4. **ダイナミクス処理**：音量変化の制御
5. **エフェクトチェーン**：複数エフェクトの組み合わせ
6. **ジャンル別プリセット**：用途に応じた設定

### 🎵 音楽制作のコツ：
- **エフェクトは適度に**：やりすぎは禁物
- **順序が重要**：エフェクトの適用順で音が大きく変わる
- **ジャンルに合わせる**：音楽スタイルに適した設定を選ぶ
- **コンテキストを考慮**：楽曲全体の中での役割を意識

### 次回予告：レッスン5
- **MIDI**システムと音楽データ
- **シーケンサー**プログラミング
- **自動作曲**の基礎
- **リズムパターン**の作成

### 🏠 宿題
1. 自分の好きな音楽ジャンル用のエフェクトプリセットを作ってみよう
2. 簡単なメロディーに3つ以上のエフェクトを組み合わせてみよう
3. エフェクトチェーンの順序を変えて、音の違いを確認してみよう